In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("data/cpi.db")

def q(sql, params=None):
    """Run a SQL query and return the result as a DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)

q("SELECT name FROM sqlite_master WHERE type = 'table'")

In [ ]:
q("PRAGMA table_info(cpi)")

In [ ]:
q("SELECT * FROM cpi LIMIT 5")

**Anatomy of a query** (the clauses always appear in this order):

```
SELECT   which columns
FROM     which table
WHERE    which rows (a condition)
ORDER BY how to sort
LIMIT    how many rows
```
- `SELECT *` means "all columns". It's fine for a quick look, but name your columns in real work.
- Text values use **single quotes**: `'All India'`. Double quotes are for column names.
- `=` tests equality (one equals sign, unlike Python's `==`).
- SQL keywords are not case sensitive, but write them in capitals so queries are easier to read.

## Part 2 – Filtering (WHERE)
**Worked example 1.** All India, Combined sector, CPI (General), 2026, in calendar order.
Predict first: how many rows? (You have seen this series before.)

In [ ]:
q("""
SELECT month, inflation
FROM cpi
WHERE state = 'All India'
  AND sector = 'Combined'
  AND division = 'CPI (General)'
  AND year = 2026
ORDER BY month_num
""")



**Worked example 2.** More ways to filter: `IN`, `BETWEEN`, `LIKE`, `DISTINCT`.

In [ ]:
q("""
SELECT DISTINCT division, code
FROM cpi
ORDER BY code
""")

In [ ]:
# DISTINCT removes repeated rows. LIKE with % is a wildcard: any characters.
q("""
SELECT DISTINCT division
FROM cpi
WHERE division LIKE '%Food%' OR division LIKE '%Health%'
""")

In [ ]:
# IN is a tidy way to write several OR conditions. BETWEEN includes both ends.
q("""
SELECT state, month, inflation
FROM cpi
WHERE year = 2026 AND month_num BETWEEN 7 AND 9
  AND sector = 'Combined'
  AND division = 'CPI (General)'
  AND state IN ('Telangana', 'Puducherry', 'Tamil Nadu')
ORDER BY state, month_num
""")

**Worked example 3 – NULL (missing values).** In SQL, `NULL` means "unknown". Unknown is never equal to anything, not even to another unknown,
the same reason `np.nan == np.nan` is False. Compare the two queries below, and predict their results first.
`COUNT(*)` simply counts rows (full aggregation arrives on Day 8).

In [ ]:
print(q("SELECT COUNT(*) AS wrong_way FROM cpi WHERE inflation = NULL"))
print(q("SELECT COUNT(*) AS right_way FROM cpi WHERE inflation IS NULL"))

## Part 3 – JOINs
A JOIN combines rows from two tables where a key matches. It is the SQL version of `merge`.

```
FROM cpi AS c
INNER JOIN zones AS z ON c.state = z.state
```
`AS c` gives the table a short alias, so `c.state` means "the state column of cpi". `ON` says how to match.

**The key difference (this is the interview question):**
- `INNER JOIN` keeps only rows that match in **both** tables.
- `LEFT JOIN` keeps **all** rows of the left table, and fills NULL where the right table has no match.

**Worked example 4.** Predict the two counts before you run them. The cpi table has 30,030 rows; the zones table has 36 states.

In [ ]:
print(q("""
SELECT COUNT(*) AS inner_join_rows
FROM cpi AS c
INNER JOIN zones AS z ON c.state = z.state
"""))
print(q("""
SELECT COUNT(*) AS left_join_rows
FROM cpi AS c
LEFT JOIN zones AS z ON c.state = z.state
"""))

The INNER JOIN silently dropped rows. **Which ones, and why?** A join that quietly loses data is the most dangerous kind of error,
so always compare row counts before and after.

**Worked example 5 – the anti-join (a validation query).** `LEFT JOIN ... WHERE right_key IS NULL` finds rows with no match.

In [ ]:
q("""
SELECT DISTINCT c.state
FROM cpi AS c
LEFT JOIN zones AS z ON c.state = z.state
WHERE z.state IS NULL
""")

**Worked example 6.** September 2026, CPI (General), Combined sector, states with their region, highest inflation first.

In [ ]:
q("""
SELECT c.state, z.region_group, c.inflation
FROM cpi AS c
INNER JOIN zones AS z ON c.state = z.state
WHERE c.year = 2026 AND c.month = 'September'
  AND c.sector = 'Combined'
  AND c.division = 'CPI (General)'
ORDER BY c.inflation DESC
LIMIT 10
""")